# fogdet: CLAHE + DCP on the TA fog dataset (Gemini 345Lg, S01 to S07)

Robot Vision & AI, Perception project, team 2. Part: CLAHE + DCP (Robin Senechal).

The TA dataset has **no detection labels**, so mAP cannot be computed here (mAP is reported on RTTS).
This notebook measures what can be measured without labels, for 3 conditions (identity = no processing, CLAHE, DCP):

| Metric | Meaning |
|---|---|
| fog score | mean dark channel (He et al.): lower = less haze. Biased in favour of DCP, which minimises it by design. |
| RMS contrast | std of the grey image, 0 to 1: higher = more contrast |
| sharpness | variance of the Laplacian: higher = sharper edges |
| defog ms / FPS | dehazing cost per frame |
| YOLO detections/frame, mean score | label-free proxy only (no ground truth; COCO has no "meat" or "icebox" class) |

Outputs in `/kaggle/working/out`: side-by-side videos (Original / CLAHE / DCP) per scenario, figures, `results.md` / `results.csv`, per-frame CSV.

**Settings:** Add the dataset `perception_fog_inference_dataset` as input. GPU is optional (only YOLO uses it). Internet must be on for the YOLO weights.

Videos and before/after figures show the YOLO boxes (COCO classes, conf ≥ 0.25) on each condition: the qualitative side-by-side comparison the guide (1-2) asks for when the dataset has no labels.

In [ ]:
# ---------------------------------------------------------------- settings
STRIDE       = 1      # process 1 frame out of STRIDE for the metrics (1 = all 7,028 pairs)
VIDEO_STRIDE = 1      # 1 frame out of N in the comparison videos (1 = full 30 fps)
YOLO_STRIDE  = 1      # run YOLO on 1 frame out of N (0 = no YOLO); boxes are drawn on those frames in the videos
MAX_FRAMES   = None   # e.g. 300 for a quick test run
PANEL_W      = 640    # width of each panel in the comparison video
SCENARIOS    = None   # e.g. ["S07_warehouse_entry_30s"]; None = all
OUT          = "/kaggle/working/out"

In [ ]:
import glob, json, os, time, subprocess
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

def find_dataset_root():
    pats = ["/kaggle/input/**/dataset/S0*_*/rgb"]
    for p in pats:
        hits = sorted(glob.glob(p, recursive=True))
        if hits:
            return Path(hits[0]).parents[1]
    raise FileNotFoundError("Dataset not found under /kaggle/input. Add 'perception_fog_inference_dataset' as input.")

DATA = find_dataset_root()
scen_dirs = sorted(d for d in DATA.iterdir() if (d / "rgb").is_dir())
if SCENARIOS:
    scen_dirs = [d for d in scen_dirs if d.name in SCENARIOS]
OUT = Path(OUT); (OUT / "videos").mkdir(parents=True, exist_ok=True); (OUT / "figures").mkdir(exist_ok=True)
print("dataset:", DATA)
for d in scen_dirs:
    print(f"  {d.name}: {len(list((d / 'rgb').glob('*.png')))} frames")

In [ ]:
# ---------------------------------------------------------------- dehazing (same code as fogdet/defog.py)
from __future__ import annotations

import sys
from pathlib import Path

import cv2
import numpy as np


class DefogBase:
    """Both input and output are (H, W, 3) uint8 BGR. Do not change this signature."""

    name = "base"

    def __call__(self, frame: np.ndarray) -> np.ndarray:
        raise NotImplementedError


# ============================================================================ identity
class IdentityDefog(DefogBase):
    """Baseline that performs no processing. Never remove it."""

    name = "identity"

    def __call__(self, frame):
        return frame


# ============================================================================ CLAHE
class ClaheDefog(DefogBase):
    """CLAHE from W1. Locally increases contrast only on the luminance channel."""

    name = "clahe"

    def __init__(self, clip=2.0, grid=8):
        self.clahe = cv2.createCLAHE(clipLimit=float(clip), tileGridSize=(int(grid), int(grid)))

    def __call__(self, frame):
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        lab[:, :, 0] = self.clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


# ============================================================================ DCP
def guided_filter(guide: np.ndarray, src: np.ndarray, radius: int, eps: float) -> np.ndarray:
    """Gray-guide guided filter (He et al., ECCV 2010), O(N) with box filters.

    guide, src: float32 (H, W) in [0, 1]. Edge-preserving smoothing of `src` following
    the edges of `guide` — removes the blocky halos of the patch-wise transmission.
    """
    ksize = (2 * radius + 1, 2 * radius + 1)

    def box(x):
        return cv2.boxFilter(x, cv2.CV_32F, ksize, borderType=cv2.BORDER_REFLECT)

    mean_i = box(guide)
    mean_p = box(src)
    corr_ip = box(guide * src)
    corr_ii = box(guide * guide)
    var_i = corr_ii - mean_i * mean_i
    cov_ip = corr_ip - mean_i * mean_p
    a = cov_ip / (var_i + eps)
    b = mean_p - a * mean_i
    return box(a) * guide + box(b)


class DcpDefog(DefogBase):
    """Dark Channel Prior (He et al., CVPR 2009). No training is required.

    1) Estimate atmospheric light A from the brightest top 0.1% pixels of the dark channel.
    2) Estimate transmission: t = 1 - omega * dark(I / A)
    3) Refine t with a guided filter and restore J = (I - A) / max(t, t0) + A.

    Args:
        patch:      dark-channel patch size (15 in the paper).
        omega:      keeps a little haze for depth perception (0.95 in the paper).
        t0:         lower bound on t, avoids noise amplification in dense fog.
        top:        fraction of the brightest dark-channel pixels used to estimate A.
        guided:     refine t with the guided filter (set False for the halo ablation).
        gf_radius:  guided-filter radius in pixels; None -> scales with image size.
        gf_eps:     guided-filter regularisation.
        a_max:      clip on A to avoid over-darkening when sky/white walls dominate.
    """

    name = "dcp"

    def __init__(self, patch=15, omega=0.95, t0=0.1, top=0.001, guided=True,
                 gf_radius=None, gf_eps=1e-3, a_max=0.98):
        self.patch, self.omega, self.t0, self.top = int(patch), float(omega), float(t0), float(top)
        self.guided, self.gf_radius, self.gf_eps, self.a_max = bool(guided), gf_radius, float(gf_eps), float(a_max)
        self._kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (self.patch, self.patch))

    def dark_channel(self, img: np.ndarray) -> np.ndarray:
        """min over RGB, then min filter (erosion) over a patch. Works on uint8 or float."""
        return cv2.erode(img.min(axis=2), self._kernel)

    def estimate_atmospheric_light(self, img: np.ndarray, dark: np.ndarray) -> np.ndarray:
        h, w = dark.shape
        n = max(1, int(h * w * self.top))
        idx = np.argpartition(dark.ravel(), -n)[-n:]
        candidates = img.reshape(-1, 3)[idx]
        # He et al. pick the single brightest candidate; the mean of the candidates is
        # more robust to isolated specular pixels and gives nearly the same A.
        A = candidates.mean(axis=0)
        return np.clip(A, 1e-3, self.a_max).astype(np.float32)

    def restore(self, frame: np.ndarray) -> dict:
        """Full DCP with intermediates (used for figures and the guided-filter ablation)."""
        img = frame.astype(np.float32) / 255.0
        dark = self.dark_channel(img)
        A = self.estimate_atmospheric_light(img, dark)
        t_raw = 1.0 - self.omega * self.dark_channel(img / A[None, None, :])
        if self.guided:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
            r = self.gf_radius or max(8, int(round(min(gray.shape) / 18)))  # ~60 px at 1080p
            t = guided_filter(gray, t_raw.astype(np.float32), int(r), self.gf_eps)
        else:
            t = t_raw
        t = np.clip(t, self.t0, 1.0)
        J = (img - A[None, None, :]) / t[..., None] + A[None, None, :]
        out = np.clip(J * 255.0 + 0.5, 0, 255).astype(np.uint8)
        return {"dark": dark, "A": A, "t_raw": t_raw, "t": t, "out": out}

    def __call__(self, frame):
        return self.restore(frame)["out"]

In [ ]:
# ---------------------------------------------------------------- label-free metrics
_K15 = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))

def _small(img, max_side=640):
    h, w = img.shape[:2]
    s = max_side / max(h, w)
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA) if s < 1 else img

def frame_metrics(bgr):
    sm = _small(bgr)
    f = sm.astype(np.float32) / 255.0
    fog = float(cv2.erode(f.min(axis=2), _K15).mean())
    gray = cv2.cvtColor(sm, cv2.COLOR_BGR2GRAY)
    contrast = float(gray.std() / 255.0)
    sharp = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    return fog, contrast, sharp

CONDITIONS = {"identity": IdentityDefog(),
              "clahe": ClaheDefog(clip=2.0, grid=8),                       # same params as the RTTS runs
              "dcp": DcpDefog(patch=15, omega=0.95, t0=0.1, guided=True)}
LABEL = {"identity": "Original (foggy)", "clahe": "CLAHE", "dcp": "DCP"}

In [ ]:
# ---------------------------------------------------------------- YOLO (optional, label-free proxy)
model = None
if YOLO_STRIDE:
    try:
        try:
            from ultralytics import YOLO
        except ImportError:
            subprocess.run(["pip", "install", "-q", "ultralytics"], check=False)
            from ultralytics import YOLO
        import torch
        DEVICE = 0 if torch.cuda.is_available() else "cpu"
        model = YOLO("yolov8n.pt")   # same detector as the RTTS results (needs Internet on)
        print("YOLO on", DEVICE)
    except Exception as e:
        print("YOLO disabled (turn Internet on in the notebook settings to enable it):", e)
        model = None

def detect(bgr):
    r = model.predict(bgr, conf=0.25, iou=0.7, imgsz=640, device=DEVICE, verbose=False)[0]
    b = r.boxes
    boxes = [(*map(int, xy), float(cf), model.names[int(c)]) for xy, cf, c in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy())]
    return len(b), (float(b.conf.mean()) if len(b) else np.nan), [x[5] for x in boxes], boxes

def draw(bgr, boxes):
    out = bgr.copy()
    t = max(2, round(min(out.shape[:2]) / 300))
    for x1, y1, x2, y2, cf, name in boxes:
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), t)
        lab = f"{name} {cf:.2f}"
        (tw, th), _ = cv2.getTextSize(lab, cv2.FONT_HERSHEY_SIMPLEX, 0.5 * t, t)
        y = max(y1, th + 4); x = min(x1, out.shape[1] - tw - 2)
        cv2.rectangle(out, (x, y - th - 4), (x + tw + 2, y), (0, 255, 0), -1)
        cv2.putText(out, lab, (x + 1, y - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.5 * t, (0, 0, 0), t, cv2.LINE_AA)
    return out

In [ ]:
# ---------------------------------------------------------------- main loop
def header(img, title, sub=""):
    bar = np.full((40, img.shape[1], 3), 25, np.uint8)
    cv2.putText(bar, title, (10, 27), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
    if sub:
        cv2.putText(bar, sub, (img.shape[1] - 10 - 9 * len(sub), 27), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (190, 190, 190), 1, cv2.LINE_AA)
    return np.vstack([bar, img])

rows, samples, classes_seen = [], {}, {}
for sd in scen_dirs:
    frames = sorted((sd / "rgb").glob("*.png"))
    if MAX_FRAMES:
        frames = frames[:MAX_FRAMES]
    meta = {}
    try:
        meta = json.loads((sd / "metadata.json").read_text())
    except Exception:
        pass
    fps_src = float(meta.get("fps", 30) or 30) if isinstance(meta, dict) else 30.0
    pick = {int(len(frames) * q) for q in (0.15, 0.5, 0.85)}
    vpath = OUT / "videos" / f"{sd.name}_compare_raw.mp4"
    writer = None
    for i, fp in enumerate(tqdm(frames, desc=sd.name)):
        do_metrics = i % STRIDE == 0
        do_video = i % VIDEO_STRIDE == 0
        if not (do_metrics or do_video or i in pick):
            continue
        img = cv2.imread(str(fp), cv2.IMREAD_COLOR)   # BGR uint8 (720, 1280, 3)
        if img is None:
            continue
        outs, shown, ndet = {}, {}, {}
        for c, fn in CONDITIONS.items():
            t0 = time.perf_counter()
            o = fn(img)
            ms = (time.perf_counter() - t0) * 1000
            outs[c] = o
            dets = None
            if model is not None and i % YOLO_STRIDE == 0:
                dets = detect(o)
                for nm in dets[2]:
                    classes_seen.setdefault((sd.name, c), {}).setdefault(nm, 0)
                    classes_seen[(sd.name, c)][nm] += 1
            shown[c] = draw(o, dets[3]) if dets else o
            ndet[c] = dets[0] if dets else None
            if do_metrics:
                fog, con, sh = frame_metrics(o)
                row = dict(scenario=sd.name, frame=fp.stem, condition=c, defog_ms=ms, fog=fog, contrast=con, sharpness=sh)
                if dets:
                    row.update(n_det=dets[0], mean_score=dets[1])
                rows.append(row)
        if i in pick:
            samples.setdefault(sd.name, []).append({c: cv2.cvtColor(_small(o, 960), cv2.COLOR_BGR2RGB) for c, o in shown.items()})
        if do_video:
            h = int(img.shape[0] * PANEL_W / img.shape[1])
            panel = np.hstack([header(cv2.resize(shown[c], (PANEL_W, h), interpolation=cv2.INTER_AREA), LABEL[c],
                                      f"{ndet[c]} det" if ndet[c] is not None else (sd.name if c == "identity" else ""))
                               for c in CONDITIONS])
            if writer is None:
                writer = cv2.VideoWriter(str(vpath), cv2.VideoWriter_fourcc(*"mp4v"), fps_src / VIDEO_STRIDE,
                                         (panel.shape[1], panel.shape[0]))
            writer.write(panel)
    if writer is not None:
        writer.release()
        final = OUT / "videos" / f"{sd.name}_compare.mp4"   # H.264 so it plays in PowerPoint
        r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(vpath), "-c:v", "libx264", "-preset", "veryfast",
                            "-crf", "23", "-pix_fmt", "yuv420p", str(final)])
        if r.returncode == 0:
            vpath.unlink()
        else:
            vpath.rename(final)

df = pd.DataFrame(rows)
df.to_csv(OUT / "per_frame.csv", index=False)
print(len(df), "rows")

In [ ]:
# ---------------------------------------------------------------- results table
def ms(s):
    s = s.dropna()
    return f"{s.mean():.3f} ± {s.std(ddof=0):.3f}" if len(s) else "—"

ORDER = list(CONDITIONS)
agg = []
for (sc, c), g in df.groupby(["scenario", "condition"], sort=False):
    base = df[(df.scenario == sc) & (df.condition == "identity")]
    d = dict(scenario=sc, condition=c, frames=len(g),
             fog=g.fog.mean(), fog_std=g.fog.std(ddof=0),
             contrast=g.contrast.mean(), contrast_std=g.contrast.std(ddof=0),
             sharpness=g.sharpness.mean(), sharpness_std=g.sharpness.std(ddof=0),
             defog_ms=g.defog_ms.mean(), fps=1000 / g.defog_ms.mean() if g.defog_ms.mean() > 0 else np.inf)
    if "n_det" in g:
        d.update(det_per_frame=g.n_det.mean(), mean_score=g.mean_score.mean())
    agg.append(d)
res = pd.DataFrame(agg)
res["condition"] = pd.Categorical(res.condition, ORDER, ordered=True)
res = res.sort_values(["scenario", "condition"])
res.to_csv(OUT / "results.csv", index=False)

lines = ["# CLAHE + DCP on the TA fog dataset (no labels)\n",
         "mean ± std over frames. fog = mean dark channel (lower = less haze; biased toward DCP). "
         "contrast = grey std / 255. sharpness = Laplacian variance. FPS = dehazing only.\n"]
has_det = "det_per_frame" in res
hdr = "| Scenario | Condition | Frames | Fog score ↓ | RMS contrast ↑ | Sharpness ↑ | Defog ms | FPS |" + (" Det/frame | Mean score |" if has_det else "")
lines += [hdr, "|" + "---|" * (hdr.count("|") - 1)]
for _, r in res.iterrows():
    line = (f"| {r.scenario} | {LABEL[r.condition]} | {r.frames} | {r.fog:.3f} ± {r.fog_std:.3f} | "
            f"{r.contrast:.3f} ± {r.contrast_std:.3f} | {r.sharpness:.0f} ± {r.sharpness_std:.0f} | "
            + (f"{r.defog_ms:.1f} | {r.fps:.1f} |" if r.condition != "identity" else "— | — |"))
    if has_det:
        line += f" {r.det_per_frame:.2f} | {r.mean_score:.3f} |"
    lines.append(line)

# all scenarios together
hdr2 = hdr.replace("| Scenario ", "")
lines += ["\n## All scenarios\n", hdr2, "|" + "---|" * (hdr2.count("|") - 1)]
for c in ORDER:
    g = df[df.condition == c]
    line = (f"| {LABEL[c]} | {len(g)} | {ms(g.fog)} | {ms(g.contrast)} | {g.sharpness.mean():.0f} ± {g.sharpness.std(ddof=0):.0f} | "
            + (f"{g.defog_ms.mean():.1f} | {1000 / g.defog_ms.mean():.1f} |" if c != "identity" else "— | — |"))
    if has_det:
        line += f" {g.n_det.mean():.2f} | {g.mean_score.mean():.3f} |"
    lines.append(line)
if classes_seen:
    lines += ["\n## Classes YOLO detected (label-free, for reference)\n"]
    for (sc, c), d in sorted(classes_seen.items()):
        top = ", ".join(f"{k} {v}" for k, v in sorted(d.items(), key=lambda x: -x[1])[:5])
        lines.append(f"- {sc} / {LABEL[c]}: {top}")
(OUT / "results.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

In [ ]:
# ---------------------------------------------------------------- figures
COL = {"identity": "#6b7280", "clahe": "#2563eb", "dcp": "#d97706"}

# 1) before / after frames per scenario
for sc, frs in samples.items():
    fig, ax = plt.subplots(len(frs), 3, figsize=(15, 2.9 * len(frs)), squeeze=False)
    for r, fr in enumerate(frs):
        for k, c in enumerate(ORDER):
            ax[r, k].imshow(fr[c]); ax[r, k].axis("off")
            if r == 0:
                ax[r, k].set_title(LABEL[c], fontsize=13)
    fig.suptitle(sc, fontsize=14); fig.tight_layout()
    fig.savefig(OUT / "figures" / f"{sc}_before_after.png", dpi=110); plt.close(fig)

# 2) metrics by scenario
metrics = [("fog", "Fog score (dark channel) ↓"), ("contrast", "RMS contrast ↑"), ("sharpness", "Sharpness (Laplacian var.) ↑")]
scs = list(dict.fromkeys(res.scenario)); x = np.arange(len(scs)); w = 0.27
fig, axs = plt.subplots(1, 3, figsize=(18, 4.6))
for ax, (m, title) in zip(axs, metrics):
    for k, c in enumerate(ORDER):
        sub = res[res.condition == c].set_index("scenario").reindex(scs)
        ax.bar(x + (k - 1) * w, sub[m], w, yerr=sub[m + "_std"], capsize=2, color=COL[c], label=LABEL[c])
    ax.set_xticks(x); ax.set_xticklabels([s.split("_")[0] for s in scs]); ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
axs[0].legend(frameon=False)
fig.tight_layout(); fig.savefig(OUT / "figures" / "metrics_by_scenario.png", dpi=130); plt.close(fig)

# 3) cost
fig, ax = plt.subplots(figsize=(6, 3.6))
CO = [c for c in ORDER if c != "identity"]
cost = df.groupby("condition").defog_ms.mean().reindex(CO)
ax.bar([LABEL[c] for c in CO], cost, color=[COL[c] for c in CO])
for i, v in enumerate(cost):
    ax.text(i, v, f"{v:.1f} ms", ha="center", va="bottom")
ax.set_ylabel("ms per 1280×720 frame"); ax.set_title("Dehazing cost"); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(OUT / "figures" / "defog_cost.png", dpi=130); plt.close(fig)
print(sorted(p.name for p in (OUT / "figures").iterdir()))

In [ ]:
# ---------------------------------------------------------------- one zip to download (Output tab)
import shutil
shutil.make_archive("/kaggle/working/fogdet_ta_results", "zip", OUT)
print("Download /kaggle/working/fogdet_ta_results.zip from the Output tab.")